You need your own api key here

In [ ]:
GOOGLE_API_KEY=""

In [ ]:
import re

def clean_text(text):
    text = re.sub(r'http\S+', 'ENLACE', text)
    text = re.sub(r'@\w+', 'USUARIO', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&gt;', '', text)
    text = re.sub(r'&lt;', '', text)
    text = re.sub(r'&#\d+;', '', text)
    text = re.sub(r'#{1,6}\s*', '', text)
    text = re.sub(r'\*{1,2}(.*?)\*{1,2}', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
import google.generativeai as genai
from google.generativeai.types import HarmCategory, HarmBlockThreshold
import time
import pandas as pd

df = pd.read_csv("../data/raw_for_LLM_labeling.csv")

genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

PROMPT_TEMPLATE = """Eres un anotador experto en análisis de sentimientos en español. 
Tu tarea es clasificar el sentimiento del siguiente texto de redes sociales.

Responde ÚNICAMENTE con una de estas tres palabras, sin explicación:
positivo
negativo
neutral

Texto: {text}

Sentimiento:"""

def label_sentiment(text, retries=3):
    cleaned = clean_text(text)
    if not cleaned:
        return "error"
    
    for attempt in range(retries): # retry loop was needed due to errors in labeling
        try:
            response = model.generate_content(
                PROMPT_TEMPLATE.format(text=cleaned[:512]),
                generation_config=genai.GenerationConfig(
                    max_output_tokens=100,
                    temperature=0,
                ),
                safety_settings={
                    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                }
            )
            if not response.candidates or not response.candidates[0].content.parts:
                time.sleep(1)
                continue  # retry

            label = response.text.strip().lower()
            if label.startswith("neg"):
                return "negativo"
            elif label.startswith("pos"):
                return "positivo"
            elif label.startswith("neu"):
                return "neutral"
            return "unknown"

        except Exception as e:
            print(f"EXCEPTION attempt {attempt+1}: {e}")
            time.sleep(1)
    
    return "error"  # all retries exhausted

C:\Users\Estanislao\AppData\Local\Temp\ipykernel_11872\2309938647.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def label_batch(texts, max_workers=5): #batch labeling with threads for speedup
    results = [None] * len(texts)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(label_sentiment, text): i 
                   for i, text in enumerate(texts)}
        for future in tqdm(as_completed(futures), total=len(texts), desc="Labeling"):
            i = futures[future]
            results[i] = future.result()
    return results

df_labeled = df.copy()
df_labeled["sentiment"] = label_batch(df_labeled["text"].tolist(), max_workers=5)

print(df_labeled["sentiment"].value_counts())
print(f"\nPer dialect:\n{df_labeled.groupby(['dialect', 'sentiment']).size().unstack()}")

df_labeled.to_csv("../data/LLM_labeled.csv", index=False)
print("Saved to data/LLM_labeled.csv")

Labeling: 100%|██████████| 5293/5293 [16:47<00:00,  5.25it/s]


sentiment
negativo    2780
neutral     1745
positivo     767
unknown        1
Name: count, dtype: int64

Per dialect:
sentiment    negativo  neutral  positivo  unknown
dialect                                          
cdmx           1326.0    785.0     328.0      1.0
madrid          163.0    170.0      65.0      NaN
rioplatense    1291.0    790.0     374.0      NaN
Saved to data/LLM_labeled.csv



negativo    2780,
neutral     1745,
positivo     767,
unknown        1

The single unknown was fixed by hand